# **Colabユーザーへの注意**

# **このファイルに直接書き込まないでください—作業が消えることがあります！**

# **必ず作業前にコピーを作成してください。**

コピーの作り方

1. 左上の「File」をクリック
> *「File」や「Runtime」などのメニューが見えないときは、右上の“v”マークを押して表示してください。*

2. 「Save a copy in Drive」を選ぶ

3. コピーしたファイル名を「YOURNAMEs_FileName.ipynb」に変更する
> 例：名前がOliviaなら → Olivias_FileName.ipynb


---

* Colabでは**30分〜90分ごとに以前の出力結果がリセットされます**。<br>
そのため、`~~ is not defined`のようなエラーが起こることがあります。<br>
その場合は、上のメニューの「Runtime」→「Run all」で最初から実行し直してください。

# 🧠 3時間で作る、自分だけのAI

このノートブックは、[Everyones_nanoGPT](https://github.com/HayatoHongo/Everyones_nanoGPT) という、本格的にLLM(大規模言語モデル)の仕組みを手を動かして学べる無料教材(全32章・所要20時間以上)から、**特に面白い部分だけを厳選した3時間の入門版**です。

### このコースのゴール

- 難しい数式は出てきません。深く理解する必要もありません。
- ゴールはただ1つ、**「自分の手でAIを作って、育てて、会話できた！」という感動を味わうこと**です。

### 今日の流れ（合計 約3時間）

| パート | 内容 | 目安時間 |
|---|---|---|
| Part 1 | 文章を数字にしてみる(トークン化) | 20分 |
| Part 2 | 小さな脳みそ(ミニAI)を組み立てる | 35分 |
| Part 3 | 自分の手でミニAIを学習させる | 25分 |
| Part 4 | 本物の大きなAIを借りてきて、会話できるように育てる | 50分 |
| Part 5 | まとめ | 10分 |

### 進め方

- 上から順にセルを実行してください(セルにカーソルを合わせて再生ボタン、またはShift+Enter)。
- 「✏️ TODO」と書かれた場所は、自分で好きな値や文章を入れてみましょう。間違えても大丈夫、AIは壊れません。
- もっと深く知りたくなったら、[本家の全32章コース](https://github.com/HayatoHongo/Everyones_nanoGPT)にぜひ挑戦してみてください。

それでは始めましょう！🚀

### ⚠️ 最初に必ずGPUに接続してください

このあとPart 2〜4で、実際にAIを学習させる作業があります。GPUなしだと10分以上かかる作業が、GPUを使えば数分で終わります。<br>
上のメニューから「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選んで保存してください。

**`Check Point`** <label><input type="checkbox">T4 GPU に接続したことを確認した</label>

In [ ]:
import torch
if torch.cuda.is_available():
    print("✅ GPUに接続されています:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPUが有効になっていません。上の手順で「ランタイムのタイプを変更」からGPU(T4など)を選んでから、このセルを再実行してください。このまま進めると、学習にとても時間がかかります。")

---
## Part 1: 文章を数字にしてみる(20分)

AI(大規模言語モデル)は、実は文字を「文字」としては理解していません。<br>
すべての文章をいったん**数字の列**に変換してから処理しています。この変換のことを**トークン化(tokenization)**と呼びます。

まずはこれを実際に体感してみましょう。

In [ ]:
# ChatGPTと同じ仲間のモデル(GPT-2)が使うトークン化の仕組みを使えるようにする
!pip -q install tiktoken

import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print("✅ 準備完了！")

### ✏️ TODO: 好きな文章を数字にしてみよう

下の `my_sentence` を、あなたの好きな文章に書き換えて実行してみましょう。

In [ ]:
my_sentence = "Hello, I am building my own AI today!"  # ✏️ TODO: 好きな文章に書き換えてみよう

token_ids = tokenizer.encode(my_sentence)
print("あなたの文章:", my_sentence)
print("AIが実際に見ている数字の列:", token_ids)
print("トークン数:", len(token_ids))

数字だけを見せられても、人間にはさっぱり意味がわかりませんね。<br>
でも安心してください。同じ道具を使えば、数字から文章に戻すこともできます。

In [ ]:
decoded_sentence = tokenizer.decode(token_ids)
print("数字から復元した文章:", decoded_sentence)

**豆知識**: 英語は1単語がだいたい1〜2トークンで済みますが、日本語や絵文字は1文字で複数トークンに分割されることがよくあります。<br>
試しに日本語の文章や絵文字を`my_sentence`に入れて、トークン数がどう変わるか見てみるのも面白いですよ。

まとめ: **AIが見ている世界は「文字」ではなく「数字の列」**。これがAIを作る上での大前提です。

---
## Part 2: 小さな脳みそ(ミニAI)を組み立てる(35分)

ここから、実際に文章を生成できるAI「ミニGPT」を、部品(クラス)を1つずつ組み合わせて作っていきます。<br>
**それぞれの部品が何をしているか、ざっくりとしたイメージだけ掴めば十分です。**中身の数式は気にしなくて大丈夫です。

| 部品 | 役割(イメージ) |
|---|---|
| TokenEmbedding | 文字を「個性を表す数字のベクトル」に変換する |
| PositionEmbedding | 「その文字が何番目に出てきたか」という情報を追加する |
| AttentionHead | 文章の中で、今の文字が「どの文字を気にすべきか」を計算する |
| MultiHeadAttention | 複数の視点(頭)で同時に「気にする相手」を探す |
| FeedForward | 集めた情報を、自分の中でいったん消化する |
| TransformerBlock | Attention と FeedForward をワンセットにしたブロック。これを何層も重ねる |
| VocabularyLogits | 最後に「次にどの文字が来そうか」の予測を出す |

これら全部を積み重ねたものが、俗に**Transformer**と呼ばれる、今のAIブームを支える仕組みです。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 練習用の文章データをダウンロードする(シェイクスピア作品の一部、文字数 約111万字)
!wget -q https://raw.githubusercontent.com/HayatoHongo/EveryonesLLM/main/input.txt -O input.txt
with open("input.txt", 'r', encoding='utf-8') as f:
    text_data = f.read()

print("✅ データ準備完了。文字数:", len(text_data))
print(text_data[:200])

In [ ]:
# 文章 <-> 数字 の変換と、学習用バッチを作るクラス
class DataLoader:
    def __init__(self, text, config):
        self.config = config
        chars = sorted(list(set(text)))  # 出てくる文字の種類を全部集める
        self.ctoi = {char: index for index, char in enumerate(chars)}  # 文字→数字
        self.itoc = {index: char for index, char in enumerate(chars)}  # 数字→文字

        self.data = torch.tensor(self.encode(text), dtype=torch.long)
        self.train_data, self.val_data = self.split_data()

    def encode(self, text):
        return [self.ctoi[c] for c in text]

    def decode(self, indices):
        return ''.join([self.itoc[i] for i in indices])

    def split_data(self):
        split_index = int(0.9 * len(self.data))  # 9割を学習用、1割を確認用に分ける
        return self.data[:split_index], self.data[split_index:]

    def get_batch(self, split):
        data = self.train_data if split == 'train' else self.val_data
        start_indices = torch.randint(len(data) - self.config.input_sequence_length, (self.config.batch_size,))
        input_sequences = torch.stack([
            data[s:s + self.config.input_sequence_length] for s in start_indices
        ])
        target_sequences = torch.stack([
            data[s + 1:s + self.config.input_sequence_length + 1] for s in start_indices
        ])
        return input_sequences.to(self.config.device_type), target_sequences.to(self.config.device_type)

print("✅ DataLoader クラス定義完了")

In [ ]:
# 文字を数字のベクトルに変換する
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, embedding_dim)

    def embed(self, input_indices):
        return self.token_embedding_table.forward(input_indices)


# 「何番目の文字か」という位置の情報を追加する
class PositionEmbedding(nn.Module):
    def __init__(self, input_sequence_length=8, embedding_dim=8):
        super().__init__()
        self.position_embedding_layer = nn.Embedding(input_sequence_length, embedding_dim)

    def forward(self, input_indices):
        sequence_length = input_indices.shape[1]
        position_indices = torch.arange(sequence_length, device=input_indices.device)
        return self.position_embedding_layer.forward(position_indices)


class EmbeddingModule(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding_layer = TokenEmbedding(vocab_size=config.vocab_size, embedding_dim=config.embedding_dim)
        self.position_embedding_layer = PositionEmbedding(input_sequence_length=config.input_sequence_length, embedding_dim=config.embedding_dim)

    def forward(self, input_indices):
        token_embeddings = self.token_embedding_layer.embed(input_indices)
        position_embeddings = self.position_embedding_layer.forward(input_indices)
        return position_embeddings + token_embeddings

print("✅ Embedding 系クラス定義完了")

In [ ]:
# 文章の中で「どの文字を気にすべきか」を計算する(Attention)
class AttentionHead(nn.Module):
    def __init__(self, head_size, config):
        super().__init__()
        self.key_fc = nn.Linear(config.embedding_dim, head_size, bias=False)
        self.query_fc = nn.Linear(config.embedding_dim, head_size, bias=False)
        self.value_fc = nn.Linear(config.embedding_dim, head_size, bias=False)
        self.dropout = nn.Dropout(config.dropout_rate)
        self.head_size = head_size

    def forward(self, input_tensor):
        B, T, C = input_tensor.shape

        Key = self.key_fc.forward(input_tensor)
        Query = self.query_fc.forward(input_tensor)
        Value = self.value_fc.forward(input_tensor)

        # ✏️ TODO: 数値を安定させるため、内積を sqrt(head_size) で割る(= -0.5乗をかける)
        attention_weights_before_mask = Query @ Key.transpose(-2, -1) * self.head_size ** (-0.5)  # ✏️ TODO: -0.5 で合っているか確認しよう

        # 未来の文字を覗き見しないようにマスクする
        mask = torch.triu(torch.ones(T, T), diagonal=1).to(input_tensor.device)
        masked_attention_weights = attention_weights_before_mask.masked_fill(mask == 1, float('-inf'))

        attention_weights = F.softmax(masked_attention_weights, dim=-1)
        attention_weights = self.dropout(attention_weights)

        out = attention_weights @ Value
        return out


# 複数の視点(頭)で同時に「気にする相手」を探す
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_attention_heads = config.num_attention_heads
        self.embedding_dim = config.embedding_dim
        self.head_size = int(self.embedding_dim / self.num_attention_heads)

        self.attention_heads = nn.ModuleList([
            AttentionHead(self.head_size, config) for _ in range(self.num_attention_heads)
        ])
        self.output_projection = nn.Linear(self.embedding_dim, self.embedding_dim)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(self, input_tensor):
        head_outputs_list = [head.forward(input_tensor) for head in self.attention_heads]
        concatenated = torch.cat(head_outputs_list, dim=-1)
        projected = self.output_projection.forward(concatenated)
        return self.dropout.forward(projected)

print("✅ Attention 系クラス定義完了")

In [ ]:
# 集めた情報を自分の中でいったん消化する
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.embedding_dim, config.hidden_dim),
            nn.ReLU(),
            nn.Linear(config.hidden_dim, config.embedding_dim),
            nn.Dropout(config.dropout_rate),
        )

    def forward(self, input_tensor):
        return self.net(input_tensor)


# Attention と FeedForward をワンセットにしたブロック
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(config.embedding_dim)
        self.layer_norm2 = nn.LayerNorm(config.embedding_dim)
        self.multihead_attention = MultiHeadAttention(config=config)
        self.feed_forward = FeedForward(config=config)

    def forward(self, input_tensor):
        normed_input = self.layer_norm1(input_tensor)
        attention_output = self.multihead_attention(normed_input)
        residual_attention = attention_output + input_tensor
        normed_attention = self.layer_norm2(residual_attention)
        feedforward_output = self.feed_forward(normed_attention)
        return feedforward_output + residual_attention


# 最後に「次にどの文字が来そうか」を予測する
class VocabularyLogits(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.output_norm = nn.LayerNorm(config.embedding_dim)
        self.vocab_projection = nn.Linear(config.embedding_dim, config.vocab_size)

    def forward(self, transformer_block_output):
        normalized_output = self.output_norm.forward(transformer_block_output)
        return self.vocab_projection.forward(normalized_output)

print("✅ FeedForward / TransformerBlock / VocabularyLogits クラス定義完了")

In [ ]:
# 1文字ずつ予測して文章を生成する関数
def generate(model, input_indices, max_new_tokens, temperature=1.0):
    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            input_conditioned = input_indices[:, -model.config.input_sequence_length:]
            logits, _ = model(input_conditioned, target_indices=None)
            last_logits = logits[:, -1, :] / temperature
            probs = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_indices = torch.cat((input_indices, next_token), dim=1)
    model.train()
    return input_indices


# ここまでの部品を全部積み重ねた、ミニAI本体
class MiniGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = EmbeddingModule(config=config)
        self.blocks = nn.Sequential(*[TransformerBlock(config=config) for _ in range(config.layer_count)])
        self.vocab_projection = VocabularyLogits(config=config)
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, input_indices, target_indices):
        embeddings = self.embedding(input_indices)
        blocks_output = self.blocks(embeddings)
        logits = self.vocab_projection(blocks_output)

        if target_indices is None:
            return logits, None

        batch_size, token_len, vocab_size = logits.shape
        logits = logits.view(batch_size * token_len, vocab_size)
        targets = target_indices.view(batch_size * token_len)
        loss = self.criterion(logits, targets)
        return logits, loss

print("🎉 MiniGPT クラス定義完了！これがあなたのAIの設計図です。")

### ✏️ TODO: あなたのAIの「脳の大きさ」を決めよう

下の数値は、AIの脳の大きさ(パラメータ数)を決める設定です。<br>
大きくするほど賢くなれる可能性がありますが、学習に時間がかかります。今回はこのままでも十分ですが、好きな値に変えても構いません。<br>
**`embedding_dim`は `num_attention_heads`(4)の倍数にしてください**(例: 32, 64, 128, 256)。`hidden_dim`は自由な値でOKです。

In [ ]:
class Config:
    batch_size = 32
    input_sequence_length = 64
    vocab_size = 65  # あとで実際の文字数に合わせて自動調整されます
    total_steps = 3000  # 学習を何回繰り返すか
    evaluation_frequency = 500
    learning_rate = 3e-3
    device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
    evaluation_loops = 5
    embedding_dim = 64        # ✏️ TODO: 好きな値に変えてみよう(4の倍数。例: 32, 64, 128, 256)
    hidden_dim = 256          # ✏️ TODO: 好きな値に変えてみよう(embedding_dimの4倍くらいが目安)
    num_attention_heads = 4
    layer_count = 4
    dropout_rate = 0.1
    random_seed_value = 1337

config = Config()
assert config.embedding_dim % config.num_attention_heads == 0, \
    f"embedding_dim({config.embedding_dim})は、num_attention_heads({config.num_attention_heads})の倍数にしてください。"

torch.manual_seed(config.random_seed_value)
print("使用デバイス:", config.device_type)

In [ ]:
data_loader = DataLoader(text_data, config)
config.vocab_size = len(data_loader.ctoi)  # 実際にデータに出てくる文字の種類数に合わせる
print("文字の種類数(vocab_size):", config.vocab_size)

model = MiniGPT(config=config).to(config.device_type)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

param_count = sum(p.numel() for p in model.parameters())
print(f"🧠 あなたのAIのパラメータ数: {param_count:,} 個")

**豆知識**: ChatGPTのようなAIは数千億パラメータ規模と言われています。<br>
今作ったミニAIはその数十万分の1以下のサイズですが、仕組みは全く同じです。この後、実際に学習させて動かしてみましょう。

---
## Part 3: 自分の手でミニAIを学習させる(25分)

学習とは、ざっくり言うと**「AIに正解を見せて、間違いを何度も直させる」**作業です。

まずは、学習前のAI(パラメータがまだランダムな状態)に文章を書かせてみましょう。

In [ ]:
prompt = "ROMEO:"  # ✏️ TODO: 好きな書き出しに変えてみよう(input.txtに出てくる文字だけ使えます)

context = torch.tensor(data_loader.encode(prompt), dtype=torch.long).unsqueeze(0).to(config.device_type)
before_training = generate(model, context, max_new_tokens=200)
print("=== 学習前のAIが書いた文章 ===")
print(data_loader.decode(before_training[0].tolist()))

意味不明な文字列が出てきたはずです。当然です、AIはまだ何も学んでいません。<br>
ここから、実際にシェイクスピア作品の文章を読ませて学習させます。**数十秒〜数分で終わります。**

In [ ]:
print("=== 学習スタート ===")

for step in range(1, config.total_steps + 1):
    input_batch, target_batch = data_loader.get_batch('train')

    optimizer.zero_grad()
    logits, loss = model(input_batch, target_batch)
    loss.backward()
    optimizer.step()

    if step % config.evaluation_frequency == 0:
        print(f"step {step:5d} | loss {loss.item():.4f}  (数字が小さいほど、正解に近づいています)")

print("=== 学習完了！ ===")

ロス(loss)の数字が、学習が進むにつれて小さくなっていくのが見えたはずです。<br>
これは「AIが少しずつ正解に近づいている」ことを意味します。

それでは、同じプロンプトで、学習後のAIに文章を書かせてみましょう。

In [ ]:
context = torch.tensor(data_loader.encode(prompt), dtype=torch.long).unsqueeze(0).to(config.device_type)
after_training = generate(model, context, max_new_tokens=300)
print("=== 学習後のAIが書いた文章 ===")
print(data_loader.decode(after_training[0].tolist()))

どうでしたか？完璧な英文ではないかもしれませんが、実在する単語やシェイクスピアっぽい言い回しが出てきたのではないでしょうか。

**たった数分前まで意味不明な記号を吐き出していたAIが、あなたの手によって「文章っぽいもの」を生成できるようになりました。**<br>
これが、ChatGPTのような巨大AIの中で起きていることと、原理的には全く同じ仕組みです。

---
## Part 4: 本物の大きなAIを借りてきて、会話できるように育てる(50分)

Part 2〜3で作ったのは、いわば「ミニチュア版」です。仕組みは全く同じまま、規模をとても大きくして、とても長い時間・とても大量の文章で学習させると、本格的なAIになります。

今回は、この教材の制作者(Hayato Hongo氏)が実際に数日かけて大規模な文章データで事前学習させた**本物のAI(パラメータ数 約5億個、ミニAIの2000倍以上)**を借りてきます。<br>
そのままでは「質問に答える」ことはまだできないので、最後に少しだけ、自分の手で「会話できるように」育てる作業(instruction tuning)を行います。

(最初にGPUへ接続済みのはずですが、まだの場合は上のメニューの「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** から接続してください。)

### Hugging Face の準備(初回のみ、5分程度)

本物のAIの重み(パラメータ)や学習データは、Hugging Face というサービスに置かれています。ダウンロードするために、無料アカウントとアクセス用のトークンが必要です。

1. [huggingface.co](https://huggingface.co/join) で無料アカウントを作成する
2. 右上のアイコン →「Settings」→「Access Tokens」→「New token」(権限は Read でOK)でトークンを作成する
3. 作成したトークンをコピーしておく

下のセルを実行すると入力欄が出るので、そこにトークンを貼り付けてください。

In [ ]:
!pip -q install huggingface_hub

from huggingface_hub import login
login()

### 本物のAIの設計図を読み込む

Part 2で作ったミニAIとほぼ同じ部品構成です(位置の表し方だけ少し違う工夫がされています)。<br>
細部を理解する必要はありません。「さっき作ったのと似た仕組みが、少し発展した形で使われている」とだけ感じてもらえれば十分です。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, embedding_dim)

    def embed(self, input_indices):
        return self.token_embedding_table.forward(input_indices)


# 相対的な位置関係(どれくらい離れた文字か)を表す、本物のAIで使われている工夫
class RelativePositionEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.relative_position_count = 2 * config.input_sequence_length - 1
        self.bias_embedding_table = nn.Embedding(self.relative_position_count, 1)

    def forward(self, query_len, key_len, device_type=None):
        query_positions = torch.arange(query_len, device=device_type)[:, None]
        key_positions = torch.arange(key_len, device=device_type)[None, :]
        relative_position_matrix = query_positions - key_positions
        relative_position_indices = relative_position_matrix + key_len - 1
        relative_position_bias_embeddings = self.bias_embedding_table(relative_position_indices)
        relative_position_bias_matrix = relative_position_bias_embeddings.squeeze(-1)
        return relative_position_bias_matrix


class AttentionHead(nn.Module):
    def __init__(self, head_size, config):
        super().__init__()
        self.key_fc = nn.Linear(config.embedding_dim, head_size, bias=False)
        self.query_fc = nn.Linear(config.embedding_dim, head_size, bias=False)
        self.value_fc = nn.Linear(config.embedding_dim, head_size, bias=False)
        self.head_size = head_size
        self.relative_position_embedding_layer = RelativePositionEmbedding(config=config)

    def forward(self, input_tensor):
        B, T, C = input_tensor.shape

        Key = self.key_fc.forward(input_tensor)
        Query = self.query_fc.forward(input_tensor)
        Value = self.value_fc.forward(input_tensor)

        attention_weights_before_mask = Query @ Key.transpose(-2, -1) * self.head_size ** (-0.5)

        relative_position_bias_matrix = self.relative_position_embedding_layer(T, T, device_type=input_tensor.device)
        attention_weights_before_mask = attention_weights_before_mask + relative_position_bias_matrix

        mask = torch.triu(torch.ones(T, T), diagonal=1).to(input_tensor.device)
        masked_attention_weights = attention_weights_before_mask.masked_fill(mask == 1, float('-inf'))

        attention_weights = F.softmax(masked_attention_weights, dim=-1)
        out = attention_weights @ Value
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.num_attention_heads = config.num_attention_heads
        self.embedding_dim = config.embedding_dim
        self.head_size = int(self.embedding_dim / self.num_attention_heads)

        self.attention_heads = nn.ModuleList([
            AttentionHead(self.head_size, config) for _ in range(self.num_attention_heads)
        ])
        self.output_projection = nn.Linear(self.embedding_dim, self.embedding_dim)

    def forward(self, input_tensor):
        head_outputs_list = [head.forward(input_tensor) for head in self.attention_heads]
        concatenated = torch.cat(head_outputs_list, dim=-1)
        projected = self.output_projection.forward(concatenated)
        return projected


class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.embedding_dim, config.hidden_dim),
            nn.ReLU(),
            nn.Linear(config.hidden_dim, config.embedding_dim),
        )

    def forward(self, input_tensor):
        return self.net(input_tensor)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(config.embedding_dim)
        self.layer_norm2 = nn.LayerNorm(config.embedding_dim)
        self.multihead_attention = MultiHeadAttention(config=config)
        self.feed_forward = FeedForward(config=config)

    def forward(self, input_tensor):
        normed_input = self.layer_norm1(input_tensor)
        attention_output = self.multihead_attention(normed_input)
        residual_attention = attention_output + input_tensor
        normed_attention = self.layer_norm2(residual_attention)
        feedforward_output = self.feed_forward(normed_attention)
        return feedforward_output + residual_attention


class VocabularyLogits(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.output_norm = nn.LayerNorm(config.embedding_dim)
        self.vocab_projection = nn.Linear(config.embedding_dim, config.vocab_size)

    def forward(self, transformer_block_output):
        normalized_output = self.output_norm.forward(transformer_block_output)
        vocab_logits = self.vocab_projection.forward(normalized_output)
        return vocab_logits


def generate(model, input_indices, max_new_tokens, temperature=1.0):
    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            input_conditioned = input_indices[:, -model.config.input_sequence_length:]
            logits, _ = model(input_conditioned, target_indices=None)
            last_logits = logits[:, -1, :] / temperature
            probs = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_indices = torch.cat((input_indices, next_token), dim=1)
    return input_indices


class EveryonesGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding_layer = TokenEmbedding(vocab_size=config.vocab_size, embedding_dim=config.embedding_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(config=config) for _ in range(config.layer_count)])
        self.vocab_projection = VocabularyLogits(config=config)
        # target が -100 の場所は損失を計算しない(instruction tuning で使う)
        self.criterion = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, input_indices, target_indices):
        token_embeddings = self.token_embedding_layer.embed(input_indices)
        blocks_output = self.blocks(token_embeddings)
        logits = self.vocab_projection(blocks_output)

        if target_indices is None:
            return logits, None

        batch_size, token_len, vocab_size = logits.shape
        logits = logits.view(batch_size * token_len, vocab_size)
        targets = target_indices.view(batch_size * token_len)
        loss = self.criterion(logits, targets)
        return logits, loss

print("✅ 本物のAIの設計図(EveryonesGPT)を定義しました")

In [ ]:
class Config:
    vocab_size = 50257
    input_sequence_length = 1024
    embedding_dim = 1280
    hidden_dim = 5120
    num_attention_heads = 10
    layer_count = 20
    device_type = 'cuda' if torch.cuda.is_available() else 'cpu'

config = Config()
model = EveryonesGPT(config=config)

param_count = sum(p.numel() for p in model.parameters())
print(f"🧠 本物のAIのパラメータ数: {param_count / 1e6:.1f} M (百万) 個")

### 本物の重み(すでに学習済みのパラメータ)をダウンロードする

上で作ったのはまだ「空っぽの脳」です。ここに、実際に長時間かけて学習させた重みを流し込みます。数GBあるので、少し時間がかかります。

In [ ]:
from huggingface_hub import hf_hub_download

model_repo_id = "HayatoHongo/EveryonesGPT-checkpoints"

# どのステップまで学習が保存されているか分からないため、大きい方から順に試す
candidate_filenames = [
    "checkpoints/checkpoint_100000.pt",
    "checkpoints/checkpoint_090000.pt",
    "checkpoints/checkpoint_050000.pt",
    "checkpoints/checkpoint_010000.pt",
]

checkpoint_path = None
for filename in candidate_filenames:
    try:
        checkpoint_path = hf_hub_download(repo_id=model_repo_id, repo_type="model", filename=filename, local_dir=".")
        print(f"✅ ダウンロード成功: {filename}")
        break
    except Exception:
        print(f"⏭️  {filename} は見つかりませんでした。次の候補を試します。")

if checkpoint_path is None:
    raise RuntimeError("チェックポイントが1つも見つかりませんでした。講師に確認してください。")

In [ ]:
checkpoint_data = torch.load(checkpoint_path, map_location="cpu")
state_dict = checkpoint_data["model_state_dict"]

# 保存時に torch.compile が使われていたため、キー名の先頭についた印を取り除く
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}

model.load_state_dict(state_dict)
model.to(config.device_type)
print("✅ 事前学習済みの重みを読み込みました！これで「空っぽの脳」に本物の知識が入りました。")

読み込めたか、実際に文章を生成させて確認してみましょう。

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

model.eval()
prompt = "The capital of Japan is"
encoded = tokenizer.encode(prompt)
encoded_tensor = torch.tensor(encoded, dtype=torch.long).unsqueeze(0).to(config.device_type)

generated = generate(model, encoded_tensor, max_new_tokens=60, temperature=0.8)
print(tokenizer.decode(generated[0].tolist()))

それらしい英文が続いたはずです。ミニAIとは比べ物にならないくらい流暢ですね。<br>
ただし、このAIは大量のWeb文章を読んで「文章の続きを書く」練習しかしていないので、**質問しても答えてくれず、ブログのように文章を続けてしまいます。**

次はこのAIを、あなたの手で「質問に答えられる」ように少しだけ育てます。これが**instruction tuning(指示に従うための追加学習)**です。

### 会話できるように、自分の手で少しだけ追加学習する

やることはシンプルです。「質問と、その模範解答」のペアをたくさん見せて、**質問文の部分は無視し、回答文の部分だけを覚えさせます。**

```
<USER>質問文<AI>回答文
      ^^^^^^^^^^^^ ここは無視(lossを計算しない)
                   ^^^^^^ ここだけ学習する
```

まずは学習データをダウンロードします。

In [ ]:
from huggingface_hub import hf_hub_download

data_path = hf_hub_download(
    repo_id="HayatoHongo/Magpie-Phi3-Pro-1M-v0.1",
    repo_type="dataset",
    filename="sft_prompt_response_phi3.jsonl",
    local_dir=".",
)
print("✅ 会話データをダウンロードしました。")

In [ ]:
import json

PAD_TOKEN_ID = tokenizer.eot_token  # 50256, 文章の終わりを表すトークン
IGNORE_INDEX = -100                 # この値の場所は損失計算をスキップする

NUM_SAMPLES = 2000  # 使うサンプル数(100万件のうち、ほんの一部だけで十分)
MAX_LEN = 256        # 学習に使う最大トークン長(短くして高速化)

input_ids_list = []
target_ids_list = []

with open(data_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= NUM_SAMPLES:
            break
        sample = json.loads(line)

        prompt_text = "<USER>" + sample["prompt"] + "<AI>"
        response_text = sample["response"] + "<|endoftext|>"

        prompt_ids = tokenizer.encode(prompt_text)
        response_ids = tokenizer.encode(response_text, allowed_special="all")

        chunk = prompt_ids + response_ids
        input_ids = chunk[:-1]

        # プロンプト部分を -100 にして、損失計算から除外する
        prompt_masked_ids = [IGNORE_INDEX] * len(prompt_ids)
        prompt_masked_chunk = prompt_masked_ids + response_ids
        target_ids = prompt_masked_chunk[1:]

        # 長すぎるサンプルは切り捨てる
        input_ids = input_ids[:MAX_LEN]
        target_ids = target_ids[:MAX_LEN]

        # 短いサンプルは埋める(padding)
        padding_length = MAX_LEN - len(input_ids)
        input_ids = input_ids + [PAD_TOKEN_ID] * padding_length
        target_ids = target_ids + [IGNORE_INDEX] * padding_length

        input_ids_list.append(input_ids)
        target_ids_list.append(target_ids)

input_ids_tensor = torch.tensor(input_ids_list, dtype=torch.long)
target_ids_tensor = torch.tensor(target_ids_list, dtype=torch.long)
print("学習データの形:", input_ids_tensor.shape, "(サンプル数, トークン長)")

### ✏️ TODO: 学習率を決めよう

instruction tuning では、事前学習のときより**ずっと小さい学習率**を使うのが定石です。<br>
学習率が大きすぎると、せっかく事前学習で身につけた知識を忘れてしまうことがあるからです。事前学習では `2e-4` が使われていたので、その1/10程度を目安にしてみましょう。

In [ ]:
learning_rate = 2e-5  # ✏️ TODO: 好きな値に変えてみよう(事前学習の2e-4の1/10くらいがおすすめ)
finetune_steps = 300
finetune_batch_size = 8

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.0)

def get_finetune_batch(batch_size):
    data_size = len(input_ids_tensor)
    indices = torch.randint(data_size, (batch_size,))
    input_batch = input_ids_tensor[indices].to(config.device_type)
    target_batch = target_ids_tensor[indices].to(config.device_type)
    return input_batch, target_batch

print("✅ 追加学習の準備完了。次のセルでいよいよ学習を始めます(数分かかります)。")

In [ ]:
model.train()
print("=== 追加学習(instruction tuning)スタート ===")

for step in range(1, finetune_steps + 1):
    input_batch, target_batch = get_finetune_batch(finetune_batch_size)

    optimizer.zero_grad()
    with torch.autocast(device_type=config.device_type, dtype=torch.bfloat16):
        logits, loss = model(input_batch, target_batch)
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(f"step {step:4d} | loss {loss.item():.4f}")

print("=== 追加学習完了！ ===")

### 🎉 いよいよチャットしてみましょう

自分の手で育てたAIに、実際に質問してみましょう。

In [ ]:
def chat_generate(model, prompt_text, max_new_tokens=100, temperature=0.7):
    model.eval()
    encoded = tokenizer.encode(prompt_text, allowed_special="all")
    input_ids = torch.tensor(encoded, dtype=torch.long, device=config.device_type).unsqueeze(0)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            input_conditioned = input_ids[:, -config.input_sequence_length:]
            logits, _ = model(input_conditioned, target_indices=None)
            last_logits = logits[:, -1, :] / temperature
            probs = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            if next_token.item() == PAD_TOKEN_ID:
                break
            input_ids = torch.cat((input_ids, next_token), dim=1)

    model.train()
    generated_ids = input_ids[0].tolist()[len(encoded):]
    return tokenizer.decode(generated_ids)


my_question = "What is your favorite season and why?"  # ✏️ TODO: 好きな質問に変えてみよう

prompt = f"<USER>{my_question}<AI>"
answer = chat_generate(model, prompt)

print("あなたの質問:", my_question)
print("AIの回答:", answer)

**`Check Point`** <label><input type="checkbox">自分で書いた質問にAIが答えてくれるのを確認した</label>

さっきまで「ブログの続きを書くだけ」だったAIが、あなたの手による数百ステップの追加学習だけで、質問に答えてくれるようになりました。<br>
これは規模こそ違えど、ChatGPTのような商用AIが作られているのと**全く同じ原理**です。

いろいろな質問を`my_question`に入れて、何度か試してみましょう。

---
## Part 5: まとめ(10分)

お疲れ様でした！今日は次のようなことを、すべて自分の手で体験しました。

1. 📐 **文章を数字にする**(トークン化)ことで、AIが世界をどう見ているかを体感した
2. 🧠 **ミニAI(Transformer)を部品から組み立てて**、その仕組みの全体像に触れた
3. 🌱 **ゼロから学習させ**、意味不明な文字列が徐々に文章らしくなる瞬間を目撃した
4. 🚀 **本物の大きなAIを借りてきて、自分の手でinstruction tuningし**、実際に会話できるようにした

**「AIを作る」というのは、魔法でも謎の技術でもなく、今日体験したような地道な仕組みの積み重ねです。**

### もっと知りたくなったら

今日のコースは、全32章・20時間以上ある[Everyones_nanoGPT](https://github.com/HayatoHongo/Everyones_nanoGPT)という本格教材のダイジェスト版です。<br>
今日省略した内容には、たとえばこんな面白いトピックがあります。

- Attentionの中身を数式レベルで完全に理解する
- 学習率のスケジューリングやチェックポイントの管理
- RoPEなど、実際の最新モデルで使われている位置情報の工夫
- 画像を理解できるAI(Vision Language Model)への拡張
- 「質問していないのに質問文を生成し始める」という不思議な現象(Magpie)

興味が湧いたら、ぜひ本家のフルコースに挑戦してみてください。

**⚠️ 右上の 🔽 からランタイムを接続解除してクレジット消費を止めてください。** <label><input type="checkbox">接続解除した</label>

**3時間クラッシュコース: 完走おめでとうございます！** <label><input type="checkbox"> Mark as Done</label>